# SCC0957 — Prática de Ciências de Dados II: Dados da Dengue PySuS
* Vinícius de Moraes - 13749910
* Rômulo Ferreira da Silva - 13734326
* João Pedro Barbosa Madeira - 13683038
* Pedro Silva dos Santos - 12688431
* Thiago Pasquotto Tavares - 15490194

## Carregando as bibliotecas

In [1]:
!uv pip install pysus==1.0.1 -q
!uv pip install nbformat
!uv pip install plotly
!uv pip install matplotlib
!uv pip install geopandas
!uv pip install requests

Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 5ms
Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 2ms
Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 7ms
Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 3ms
Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 3ms


In [2]:
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.express as px
from pysus import SINAN
import polars as pl
import geopandas as gpd
import requests
from pysus.online_data import IBGE

## Carregando os dados

### 1. Carregando os Dados do SINAN

In [3]:
sinan = SINAN().load() # Loads the files from DATASUS
files = sinan.get_files(dis_code=["DENG"])
parquet = sinan.download(files)

22316488it [00:00, 336189205.86it/s]  


In [4]:
df = pl.scan_parquet(
    [str(p) for p in parquet],
    extra_columns="ignore",
    missing_columns="insert"
)

In [5]:
df.collect_schema()

Schema([('ID_MUNICIP', String),
        ('ID_UNIDADE', String),
        ('DT_NOTIFIC', String),
        ('CS_RACA', String),
        ('CS_ESCOLAR', String),
        ('NU_ANO', String),
        ('SEM_NOT', String),
        ('SG_UF_NOT', String),
        ('ID_REGIONA', String),
        ('DT_SIN_PRI', String),
        ('SEM_PRI', String),
        ('NU_IDADE', String),
        ('CS_SEXO', String),
        ('ID_MN_RESI', String),
        ('ID_RG_RESI', String),
        ('SG_UF', String),
        ('ID_PAIS', String),
        ('ID_DG_NOT', String),
        ('ID_EV_NOT', String),
        ('ANT_DT_INV', String),
        ('OCUPACAO', String),
        ('DENGUE', String),
        ('ANO', String),
        ('VACINADO', String),
        ('DT_DOSE', String),
        ('FEBRE', String),
        ('DT_FEBRE', String),
        ('DURACAO', String),
        ('LACO', String),
        ('CEFALEIA', String),
        ('EXANTEMA', String),
        ('DOR', String),
        ('PROSTACAO', String),
        ('MIALGIA',

### 2. Carregando os Dados Geográficos do Geodata

In [6]:
url = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"

gdf = gpd.read_file(url)

gdf['latitude'] = gdf.geometry.centroid.y
gdf['longitude'] = gdf.geometry.centroid.x

gdf

/tmp/ipykernel_6489/2435523726.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['latitude'] = gdf.geometry.centroid.y
/tmp/ipykernel_6489/2435523726.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['longitude'] = gdf.geometry.centroid.x


,id,name,description,geometry,latitude,longitude
0,1100015,Alta Floresta D'Oeste,Alta Floresta D'Oeste,"POLYGON ((-62.18209 -11.86686, -62.1623 -11.87...",-12.469633,-62.274095
1,1100023,Ariquemes,Ariquemes,"POLYGON ((-62.53595 -9.73182, -62.50782 -9.754...",-9.951564,-62.956923
2,1100031,Cabixi,Cabixi,"POLYGON ((-60.3994 -13.45584, -60.40195 -13.46...",-13.474432,-60.639192
3,1100049,Cacoal,Cacoal,"POLYGON ((-61.00051 -11.39796, -61.01794 -11.4...",-11.300812,-61.324237
4,1100056,Cerejeiras,Cerejeiras,"POLYGON ((-61.50047 -13.00392, -61.47901 -13.0...",-13.203228,-61.260380
...,...,...,...,...,...,...
5559,5222005,Vianópolis,Vianópolis,"POLYGON ((-48.43125 -16.62755, -48.42527 -16.6...",-16.812518,-48.441465
5560,5222054,Vicentinópolis,Vicentinópolis,"POLYGON ((-49.85005 -17.57682, -49.84311 -17.5...",-17.724391,-49.872132
5561,5222203,Vila Boa,Vila Boa,"POLYGON ((-47.11019 -14.6715, -47.11607 -14.67...",-14.992990,-47.062354
5562,5222302,Vila Propício,Vila Propício,"POLYGON ((-48.75124 -14.90461, -48.75196 -14.9...",-15.268832,-48.814195


### 3. Carregando os dados de população do IBGE

In [7]:
census = IBGE.get_population(year=2010)

In [8]:
gdf['id_6'] = gdf['id'].astype(str).str[:-1]
census = census.merge(
    gdf[['id_6', 'name']],
    left_on='MUNIC_RES',
    right_on='id_6',
    how='left'
)

census.drop(labels='id_6', axis=1, inplace=True)

In [9]:
gdf['UF_cod'] = gdf['id_6'].str[:2].astype(int)

cod_to_uf = {
    11:'RO',12:'AC',13:'AM',14:'RR',15:'PA',16:'AP',17:'TO',
    21:'MA',22:'PI',23:'CE',24:'RN',25:'PB',26:'PE',27:'AL',28:'SE',29:'BA',
    31:'MG',32:'ES',33:'RJ',35:'SP',
    41:'PR',42:'SC',43:'RS',
    50:'MS',51:'MT',52:'GO',53:'DF'
}

gdf['UF'] = gdf['UF_cod'].map(cod_to_uf)

census = census.merge(
    gdf[['id_6', 'UF']],
    left_on='MUNIC_RES',
    right_on='id_6',
    how='left'
)

census.drop(columns=['id_6'], inplace=True)

In [10]:
census = census.drop_duplicates(subset='name')

In [11]:
census

,MUNIC_RES,ANO,POPULACAO,name,UF
0,110001,2010,24422,Alta Floresta D'Oeste,RO
1,110002,2010,90354,Ariquemes,RO
2,110003,2010,6309,Cabixi,RO
3,110004,2010,78601,Cacoal,RO
4,110005,2010,17030,Cerejeiras,RO
...,...,...,...,...,...
5560,522200,2010,12549,Vianópolis,GO
5561,522205,2010,7371,Vicentinópolis,GO
5562,522220,2010,4742,Vila Boa,GO
5563,522230,2010,5145,Vila Propício,GO


## Preparando os dados 

**Número de notificações/dia**

In [12]:
result = (
    df
    .group_by("DT_NOTIFIC")
    .agg(pl.len().alias("Notificações"))
    .rename({"DT_NOTIFIC": "Dia"})
    .sort("Dia")
)

df_notificacoes = result.collect().to_pandas()

df_notificacoes['Dia'] = pd.to_datetime(df_notificacoes['Dia'], format='%Y%m%d', errors='coerce')

df_notificacoes = df_notificacoes.dropna(subset=['Dia'])

df_notificacoes

,Dia,Notificações
0,2000-01-01,13
1,2000-01-02,97
2,2000-01-03,341
3,2000-01-04,386
4,2000-01-05,422
...,...,...
9705,2026-07-29,130
9706,2026-07-30,142
9707,2026-07-31,124
9708,2026-08-01,59


**Sazonalidade dos dados**

In [13]:
df_notificacoes["Ano"] = df_notificacoes["Dia"].dt.year
df_notificacoes["Mês"] = df_notificacoes["Dia"].dt.month

df_sazonalidade = (
    df_notificacoes
    .groupby(["Ano", "Mês"])["Notificações"]
    .mean()
    .reset_index()
)

df_sazonalidade

,Ano,Mês,Notificações
0,2000,1,458.290323
1,2000,2,853.103448
2,2000,3,1037.516129
3,2000,4,963.266667
4,2000,5,795.774194
...,...,...,...
315,2026,4,2933.766667
316,2026,5,2675.419355
317,2026,6,1978.900000
318,2026,7,1292.193548


**Classificação final da doença**

In [39]:
classi_fin = (
    df
    .select("DENGUE")
    .collect()
    .to_pandas()
)

classi_fin = (
    df
    .select(
        pl.col("DENGUE")
        .str.strip_chars()
        .replace({
            "9": "Ignorado",
            "8": "Inconclusivo",
            "1": "Positivo",
            "2": "Negativo",
            "": "Ignorado"
        })
        .fill_null("Não preenchido")
        .alias("DENGUE")
    )
    .collect()
    .to_pandas()
)

print(classi_fin['DENGUE'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
classi_fin

DENGUE
Não preenchido    90.11%
Negativo           6.36%
Ignorado           2.72%
Positivo           0.81%
Name: proportion, dtype: object


,DENGUE
0,Negativo
1,Negativo
2,Negativo
3,Negativo
4,Negativo
...,...
28152067,Não preenchido
28152068,Não preenchido
28152069,Não preenchido
28152070,Não preenchido


**Notificações por sexo**

In [40]:
cs_sexo = (
    df
    .select(
        pl.col("CS_SEXO")
        .str.strip_chars()
        .replace({
            'M': 'Masculino',
            'F': 'Feminino',
            'I': 'Ignorado',
            '' : 'Ignorado',
            'O': 'Ignorado'
        })
        .fill_null("Não preenchido")
        .alias("CS_SEXO")
    )
    .collect()
    .to_pandas()
)

print(cs_sexo['CS_SEXO'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
cs_sexo

CS_SEXO
Feminino     54.96%
Masculino    44.91%
Ignorado      0.13%
Name: proportion, dtype: object


,CS_SEXO
0,Feminino
1,Masculino
2,Masculino
3,Feminino
4,Feminino
...,...
28152067,Masculino
28152068,Masculino
28152069,Feminino
28152070,Masculino


**Notificações por raça/etnia**

In [41]:
cs_raca = (
    df
    .select(
        pl.col("CS_RACA")
        .str.strip_chars()
        .replace({
            '': 'Ignorado',
            '1': 'Branca',
            '2': 'Preta',
            '3': 'Amarela',
            '4': 'Parda',
            '5': 'Indígena',
            '9': 'Ignorado',
            '@': 'Ignorado'
        })
        .fill_null("Não preenchido")
        .alias("CS_RACA")
    )
    .collect()
    .to_pandas()
)

print(cs_raca['CS_RACA'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
cs_raca

CS_RACA
Branca       34.6%
Parda       31.73%
Ignorado    28.31%
Preta        4.15%
Amarela      0.96%
Indígena     0.25%
Name: proportion, dtype: object


,CS_RACA
0,Preta
1,Branca
2,Parda
3,Parda
4,Parda
...,...
28152067,Parda
28152068,Ignorado
28152069,Branca
28152070,Branca


**Evolução dos casos**

In [42]:
df_evolucao = (
    df.select(
        pl.col('CON_EVOLUC')
        .str.strip_chars()
        .replace({
            '0': 'não se aplica',
            '1': 'cura',
            '2': 'óbito pelo agravo',
            '3': 'óbito por outras causas',
            '4': 'óbito em investigação',
            '9': 'ignorado',
            '': 'ignorado',
            ']': 'ignorado'
        })
        .fill_null('Não preenchido')
        .alias('CON_EVOLUC')
    )
    .collect()
    .to_pandas()
)

print(df_evolucao['CON_EVOLUC'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
df_evolucao

CON_EVOLUC
Não preenchido             90.11%
cura                        6.89%
ignorado                    1.94%
não se aplica               1.05%
óbito pelo agravo           0.01%
óbito por outras causas      0.0%
Name: proportion, dtype: object


,CON_EVOLUC
0,cura
1,cura
2,cura
3,cura
4,cura
...,...
28152067,Não preenchido
28152068,Não preenchido
28152069,Não preenchido
28152070,Não preenchido


**Número de notificações por estado**

In [17]:
df_estado = (
    df
    .with_columns(
        pl.col("UF")
        .cast(pl.String)
        .str.strip_chars()
        .replace({
            "11": "RO",
            "12": "AC",
            "13": "AM",
            "14": "RR",
            "15": "PA",
            "16": "AP",
            "17": "TO",
            "21": "MA",
            "22": "PI",
            "23": "CE",
            "24": "RN",
            "25": "PB",
            "26": "PE",
            "27": "AL",
            "28": "SE",
            "29": "BA",
            "31": "MG",
            "32": "ES",
            "33": "RJ",
            "35": "SP",
            "41": "PR",
            "42": "SC",
            "43": "RS",
            "50": "MS",
            "51": "MT",
            "52": "GO",
            "53": "DF",
            "0": None,
            "": None,
            "`": None
        })
    )
    .drop_nulls("UF")
    .group_by("UF")
    .agg(pl.len().alias("Notificações"))
    .rename({"UF": "Estado"})
    .sort("Estado")
    .collect()
    .to_pandas()
)

df_estado

,Estado,Notificações
0,AC,4366
1,AL,8729
2,AM,5431
3,AP,2942
4,BA,42985
5,CE,34112
6,DF,28355
7,ES,27151
8,GO,90136
9,MA,27234


**Número de notificações por município** 

In [18]:
gdf['id_6'] = gdf['id'].astype(str).str[:-1]

df_municipio = (
    df
    .join(
        pl.from_pandas(
            gdf[['id_6', 'name', 'latitude', 'longitude']]
        ).lazy(),
        left_on='ID_MUNICIP',
        right_on='id_6',
        how='left'
    )
    .group_by(['name', 'latitude', 'longitude'])
    .agg(
        pl.len().alias('Notificações')
    )
    .drop_nulls('name')
    .collect()
    .to_pandas()
)

df_municipio

,name,latitude,longitude,Notificações
0,Bacabeira,-2.932518,-44.366971,459
1,Canaã,-20.666772,-42.621934,186
2,Nova Viçosa,-17.899820,-39.715332,2177
3,Arco-Íris,-21.766067,-50.430201,608
4,Nova Era,-19.718970,-43.013634,2640
...,...,...,...,...
5524,Patrocínio,-18.970652,-47.050743,16330
5525,Rosário,-2.944563,-44.201741,589
5526,Guará,-20.481951,-47.775143,2345
5527,Nova Belém,-18.488297,-41.101403,400


**Número de notificações por município por 100 mil habitantes**

In [19]:
df_municipio_relativo = df_municipio.merge(census[['POPULACAO', 'name']], on='name', how='left')

df_municipio_relativo['POPULACAO'] = (
    df_municipio_relativo['POPULACAO']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '', regex=False)
)

df_municipio_relativo['POPULACAO'] = pd.to_numeric(df_municipio_relativo['POPULACAO'], errors='coerce')

df_municipio_relativo = df_municipio_relativo.dropna(subset=['POPULACAO'])

df_municipio_relativo['taxa_100k'] = (df_municipio_relativo['Notificações'] / df_municipio_relativo['POPULACAO']) * 100000

df_municipio_relativo

,name,latitude,longitude,Notificações,POPULACAO,taxa_100k
0,Bacabeira,-2.932518,-44.366971,459,14965,3067.156699
1,Canaã,-20.666772,-42.621934,186,4631,4016.411142
2,Nova Viçosa,-17.899820,-39.715332,2177,38537,5649.116434
3,Arco-Íris,-21.766067,-50.430201,608,1925,31584.415584
4,Nova Era,-19.718970,-43.013634,2640,17540,15051.311288
...,...,...,...,...,...,...
5524,Patrocínio,-18.970652,-47.050743,16330,82541,19784.107292
5525,Rosário,-2.944563,-44.201741,589,39582,1488.050124
5526,Guará,-20.481951,-47.775143,2345,19864,11805.275876
5527,Nova Belém,-18.488297,-41.101403,400,3732,10718.113612


## Visualizações

In [20]:
media = df_notificacoes['Notificações'].mean()

fig = px.line(
    df_notificacoes,
    x='Dia',
    y='Notificações'
)

fig.add_hline(
    y=media,
    line_dash='dash',
    annotation_text=f"Média: {media:.0f}",
    annotation_position="top left"
)

fig.update_traces(line=dict(width=1))

fig.show()

In [21]:
fig = px.line(
    df_sazonalidade,
    x="Mês",
    y="Notificações",
    color="Ano",
    markers=True,
    labels={
        "Mês": "Mês",
        "Notificações": "Notificações",
        "Ano": "Ano"
    },
    title="Sazonalidade das notificações por ano"
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(1, 13)),
        ticktext=[
            "Jan", "Fev", "Mar", "Abr",
            "Mai", "Jun", "Jul", "Ago",
            "Set", "Out", "Nov", "Dez"
        ]
    )
)

fig.show()

In [22]:
from plotly.subplots import make_subplots

# Criar ano e mês
df_notificacoes["Ano"] = df_notificacoes["Dia"].dt.year
df_notificacoes["Mês"] = df_notificacoes["Dia"].dt.month

# Soma das notificações de cada mês de cada ano
df_mensal = (
    df_notificacoes
    .groupby(["Ano", "Mês"])["Notificações"]
    .sum()
    .reset_index()
)

nomes_meses = [
    "Janeiro", "Fevereiro", "Março", "Abril",
    "Maio", "Junho", "Julho", "Agosto",
    "Setembro", "Outubro", "Novembro", "Dezembro"
]

# Criar 12 gráficos
fig = make_subplots(
    rows=4,
    cols=3,
    subplot_titles=nomes_meses
)

for mes in range(1, 13):

    dados = df_mensal[df_mensal["Mês"] == mes]

    # Média histórica daquele mês
    media = dados["Notificações"].mean()

    linha = go.Scatter(
        x=dados["Ano"],
        y=dados["Notificações"],
        mode="lines+markers",
        name="Notificações",
        showlegend=False
    )

    media_linha = go.Scatter(
        x=dados["Ano"],
        y=[media] * len(dados),
        mode="lines",
        name="Média",
        line=dict(dash="dash"),
        showlegend=False
    )

    linha_idx = (mes - 1) // 3 + 1
    coluna_idx = (mes - 1) % 3 + 1

    fig.add_trace(
        linha,
        row=linha_idx,
        col=coluna_idx
    )

    fig.add_trace(
        media_linha,
        row=linha_idx,
        col=coluna_idx
    )

fig.update_layout(
    title="Tendência das notificações por mês ao longo dos anos",
    height=900,
    width=1200
)

fig.show()

In [23]:
# Contagem dos resultados
dengue_counts = (
    classi_fin["DENGUE"]
    .value_counts()
    .reset_index()
)

dengue_counts.columns = ["Resultado", "Contagem"]

fig = px.bar(
    dengue_counts,
    x="Resultado",
    y="Contagem",
    text="Contagem",
    title="Número de casos de Dengue (2000-2026)",
    labels={
        "Contagem": "Número de casos",
        "Resultado": "Resultado"
    },
    color="Resultado"
)

fig.update_traces(textposition="outside")

fig.show()

In [35]:
# Contagem dos resultados
sexo_counts = (
    cs_sexo["CS_SEXO"]
    .value_counts()
    .reset_index()
)

sexo_counts.columns = ["Sexo", "Contagem"]

fig = px.bar(
    sexo_counts,
    x="Sexo",
    y="Contagem",
    text="Contagem",
    title="Número de casos de Dengue por Sexo (2000-2026)",
    labels={
        "Contagem": "Número de casos",
        "Sexo": "Sexo"
    },
    color="Sexo"
)

fig.update_traces(textposition="outside")

fig.show()

In [24]:
# Contagem dos resultados
raca_counts = (
    cs_raca["CS_RACA"]
    .value_counts()
    .reset_index()
)

raca_counts.columns = ["Raça", "Contagem"]

fig = px.bar(
    raca_counts,
    x="Raça",
    y="Contagem",
    text="Contagem",
    title="Número de casos de Dengue por Raça (2000-2026)",
    labels={
        "Contagem": "Número de casos",
        "Raça": "Raça"
    },
    color="Raça"
)

fig.update_traces(textposition="outside")

fig.show()

In [25]:
# Contagem dos resultados
evolucao_counts = (
    df_evolucao["CON_EVOLUC"]
    .value_counts()
    .reset_index()
)

evolucao_counts.columns = ["Evolução", "Contagem"]

fig = px.bar(
    evolucao_counts,
    x="Evolução",
    y="Contagem",
    text="Contagem",
    title="Evolução dos casos de Dengue (2000-2026)",
    labels={
        "Contagem": "Número de casos",
        "Evolução": "Evolução"
    },
    color="Evolução"
)

fig.update_traces(textposition="outside")

fig.show()

In [26]:
df_estado["Percentual"] = (
    df_estado["Notificações"]
    / df_estado["Notificações"].sum()
    * 100
)

df_estado = df_estado.sort_values(
    by="Notificações",
    ascending=False
)

fig = px.bar(
    df_estado,
    x="Estado",
    y="Notificações",
    color="Notificações",
    color_continuous_scale="Reds",
    text=df_estado["Percentual"].map(lambda x: f"{x:.1f}%"),
    title="Distribuição de Casos Positivos de Dengue por Estado (2000-2026)",
)

fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Estado (UF)",
    yaxis_title="Número de Casos",
    showlegend=False
)

fig.show()

In [27]:
fig = px.scatter_mapbox(
    df_municipio,
    lat='latitude',
    lon='longitude',
    size='Notificações',
    hover_name='name',
    size_max=30,
    zoom=3,
    mapbox_style="carto-positron",
    title='Número de notificações de Dengue por município (2000-2026)',
    height=800
)

fig.update_traces(marker=dict(color='red'))

fig.show()

/tmp/ipykernel_6489/1396469008.py:1: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [28]:
fig = px.scatter_mapbox(
    df_municipio_relativo,
    lat='latitude',
    lon='longitude',
    size='taxa_100k',
    hover_name='name',
    hover_data={
        'Notificações': True,
        'POPULACAO': True,
        'taxa_100k': ':.2f'
    },
    size_max=30,
    zoom=3,
    mapbox_style="carto-positron",
    title='Taxa de casos de Dengue por 100 mil habitantes'
)

fig.update_traces(marker=dict(color='red'))

fig.update_layout(
    width=1200,
    height=800
)

fig.show()

/tmp/ipykernel_6489/2060644747.py:1: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [29]:
top_mun = df_municipio.sort_values(
    'Notificações',
    ascending=False
).head(10)

top_mun['Notificações'] = top_mun['Notificações'].astype(float)

fig = px.bar(
    top_mun.sort_values('Notificações'),
    x='Notificações',
    y='name',
    orientation='h',
    text='Notificações',
    title='Top 10 municípios com mais notificações de Dengue (2000-2026)',
    labels={
        'Notificações': 'Número de notificações',
        'name': 'Município'
    },
    color='Notificações'
)

fig.update_traces(textposition='outside')

fig.show()

In [30]:
top_mun_relativo = df_municipio_relativo.sort_values(
    'taxa_100k',
    ascending=False
).head(10)

fig = px.bar(
    top_mun_relativo.sort_values('taxa_100k'),
    x='taxa_100k',
    y='name',
    orientation='h',
    text=top_mun_relativo['taxa_100k'].round(2),
    title='Top 10 municípios com mais casos por 100 mil habitantes (2000-2026)',
    labels={
        'taxa_100k': 'Número de taxa_100k',
        'name': 'Município'
    },
    color='taxa_100k'
)

fig.update_traces(textposition='outside')

fig.show()